# Fine-tuning Gemma 4 31B with Axolotl and LoRA on HiPerGator

This classroom notebook performs supervised fine-tuning (SFT) of **Gemma 4 31B** with **BF16 LoRA** on a sanitized PHIL dataset. It is designed for a GPU-backed HiPerGator Open OnDemand Jupyter session. Run cells in order.

> **Important:** This notebook deliberately uses standard LoRA, not QLoRA: the base model and adapter use BF16 and no 4-bit/8-bit loading is configured. A 31B BF16 model needs a high-memory GPU; the preflight checks this before training.

## Learning objectives

By the end, you should be able to: (1) format instruction data as chat conversations, (2) explain why LoRA updates a small set of trainable weights, (3) configure and run Axolotl, (4) compare a base model with an adapter, and (5) recognize why loss and sample outputs are not clinical validation.

## Before class

Launch an Open OnDemand Jupyter session with one high-memory GPU (80 GB or more is the minimum target), adequate project storage, and internet access to Hugging Face. Accept the Gemma license on Hugging Face before beginning. The dataset must be de-identified and approved for classroom use.

## 1. Concepts: SFT, LoRA, Gemma, and Axolotl

In SFT, each example teaches a model to continue a conversation with a preferred assistant response. Full fine-tuning updates every model parameter. **LoRA** freezes the base model and learns small low-rank matrices in selected attention and MLP projections, reducing trainable parameters and checkpoint size. Unlike QLoRA, this lesson keeps the base model in BF16.

Gemma 4 is a modern instruction model with a chat template. Axolotl prepares chat data, applies PEFT/LoRA, manages training and evaluation, and writes resumable checkpoints. We use a small held-out split for learning purposes; it does not establish safety, accuracy, fairness, or clinical utility.

**Discuss:** What behavior should be learned from the data, and what behavior must remain unchanged?

In [ ]:
# Instructor parameters -- edit only this cell for a new class run.
from pathlib import Path

MODEL_ID = 'google/gemma-4-31b-it'  # Replace only if your approved HF model ID differs.
SOURCE_DATASET = Path('/blue/prismap-ai-core/llm_projects/digital_twin/LLM_ANKITH/dr_hus_classroom_finetuning/data/phil_sanitized.jsonl')
CLASSROOM_ROOT = Path('/blue/prismap-ai-core/llm_projects/digital_twin/LLM_ANKITH/dr_hus_classroom_finetuning')
RUN_NAME = 'gemma4_31b_lora_classroom'
MAX_SAMPLES = 1000
SEED = 42
MIN_GPU_MEMORY_GB = 80

RUN_ROOT = CLASSROOM_ROOT / 'runs' / RUN_NAME
CACHE_ROOT = CLASSROOM_ROOT / 'cache'
for directory in (RUN_ROOT, CACHE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print(f'Run artifacts: {RUN_ROOT}')
print(f'Source dataset: {SOURCE_DATASET}')

## Create your personal Python 3.11 Jupyter kernel (one time)

A notebook cannot replace its own Python interpreter while it is running. Each student must therefore create and select a personal kernel once before running the setup cells below. Run the **bootstrap cell directly below** from the initial Jupyter kernel, or use the equivalent terminal commands shown here. The environment and its installation caches live on `/blue` inside `.student_envs/<username>` in this classroom folder; only the tiny Jupyter kernel registration is written to your user Jupyter configuration. Do not copy `.student_envs` when sharing the notebook.

```bash
module load conda/26.7
ENV_ROOT="/blue/prismap-ai-core/llm_projects/digital_twin/LLM_ANKITH/dr_hus_classroom_finetuning/.student_envs/$USER"
ENV_PREFIX="$ENV_ROOT/axolotl_gemma4"
export CONDA_PKGS_DIRS="$ENV_ROOT/cache/conda_pkgs" PIP_CACHE_DIR="$ENV_ROOT/cache/pip" XDG_CACHE_HOME="$ENV_ROOT/cache/xdg"
conda create -y -p "$ENV_PREFIX" python=3.11 pip
conda run -p "$ENV_PREFIX" python -m pip install --upgrade pip
conda run -p "$ENV_PREFIX" python -m pip install 'axolotl==0.18.0' 'transformers>=5.14,<5.15' 'peft>=0.19,<0.20' datasets accelerate sentencepiece pyyaml huggingface_hub ipykernel
conda run -p "$ENV_PREFIX" python -m pip install 'torchvision==0.27.1+cu130' --index-url https://download.pytorch.org/whl/cu130
conda run -p "$ENV_PREFIX" python -m ipykernel install --user --name "axolotl-gemma4-$USER" --display-name "Python 3.11 (My Axolotl Gemma 4)"
```

After the commands finish, restart the Open OnDemand Jupyter session, choose **Kernel → Change Kernel → Python 3.11 (My Axolotl Gemma 4)**, and run this notebook from the beginning. Do not use the system `/apps/jupyter/...` Python 3.10 kernel. If the kernel does not appear, fully stop and relaunch the Jupyter session after registration.

In [ ]:
# ONE-TIME BOOTSTRAP: run this from the initial Jupyter kernel, then restart Jupyter and change kernels.
# It can take several minutes and may download large packages. Do not run it concurrently in multiple notebooks.
import getpass, os, subprocess

student_env_root = CLASSROOM_ROOT / '.student_envs' / getpass.getuser()
student_env = student_env_root / 'axolotl_gemma4'
kernel_name = f'axolotl-gemma4-{getpass.getuser()}'
bootstrap_script = r'''
set -euo pipefail
source /etc/profile.d/modules.sh 2>/dev/null || true
module load conda/26.7
mkdir -p "$STUDENT_CACHE_ROOT/conda_pkgs" "$STUDENT_CACHE_ROOT/pip" "$STUDENT_CACHE_ROOT/xdg"
export CONDA_PKGS_DIRS="$STUDENT_CACHE_ROOT/conda_pkgs"
export PIP_CACHE_DIR="$STUDENT_CACHE_ROOT/pip"
export XDG_CACHE_HOME="$STUDENT_CACHE_ROOT/xdg"
if [ ! -x "$STUDENT_ENV_PREFIX/bin/python" ]; then
  conda create -y -p "$STUDENT_ENV_PREFIX" python=3.11 pip
fi
"$STUDENT_ENV_PREFIX/bin/python" -m pip install --upgrade pip
"$STUDENT_ENV_PREFIX/bin/python" -m pip install 'axolotl==0.18.0' 'transformers>=5.14,<5.15' 'peft>=0.19,<0.20' datasets accelerate sentencepiece pyyaml huggingface_hub ipykernel
# Gemma 4 imports its processor through torchvision even for text-only training.
"$STUDENT_ENV_PREFIX/bin/python" -m pip install 'torchvision==0.27.1+cu130' --index-url https://download.pytorch.org/whl/cu130
"$STUDENT_ENV_PREFIX/bin/python" -m ipykernel install --user --name "$STUDENT_KERNEL_NAME" --display-name "Python 3.11 (My Axolotl Gemma 4)"
'''
bootstrap_env = os.environ | {'STUDENT_ENV_PREFIX': str(student_env), 'STUDENT_CACHE_ROOT': str(student_env_root / 'cache'), 'STUDENT_KERNEL_NAME': kernel_name}
subprocess.run(['bash', '-lc', bootstrap_script], env=bootstrap_env, check=True)
print('Personal kernel created:', kernel_name)
print('Now fully restart the Open OnDemand Jupyter session, then select: Python 3.11 (My Axolotl Gemma 4).')

## 2. Environment setup and preflight

HiPerGator login homes are not appropriate for large model caches. The next cells place Hugging Face, Torch, Triton, and extension caches in the classroom project directory. **Before running them, select your personal Python 3.11 Axolotl/Gemma 4 kernel in Kernel → Change Kernel.** Do not use the system `/apps/jupyter/...` Python 3.10 kernel: it is incompatible with this stack. The bootstrap cell has already installed packages; this section only validates them.

In [ ]:
import os, sys
os.environ.update({
    'HF_HOME': str(CACHE_ROOT / 'huggingface'),
    'TRANSFORMERS_CACHE': str(CACHE_ROOT / 'huggingface' / 'hub'),
    'TORCH_HOME': str(CACHE_ROOT / 'torch'),
    'TRITON_CACHE_DIR': str(CACHE_ROOT / 'triton'),
    'TORCH_EXTENSIONS_DIR': str(CACHE_ROOT / 'torch_extensions'),
    'TOKENIZERS_PARALLELISM': 'false',
})
for value in os.environ.values():
    if value.startswith(str(CACHE_ROOT)):
        Path(value).mkdir(parents=True, exist_ok=True)
print('Cache root:', CACHE_ROOT)
if sys.version_info < (3, 11):
    raise RuntimeError(
        f'This kernel is Python {sys.version.split()[0]}. Create/select your personal Axolotl/Gemma 4 Python 3.11 kernel '
        'using the instructions above, then restart from the parameter cell. Do not install Axolotl into /apps/jupyter Python 3.10.'
    )
from importlib.metadata import PackageNotFoundError, version
required_packages = ('axolotl', 'transformers', 'peft', 'datasets', 'accelerate', 'torchvision')
missing_packages = []
for package in required_packages:
    try:
        print(f'{package}: {version(package)}')
    except PackageNotFoundError:
        missing_packages.append(package)
if missing_packages:
    raise RuntimeError(f'Missing {missing_packages}. Run the one-time personal-kernel bootstrap cell, restart Jupyter, and select the new kernel.')
print('Personal kernel dependencies are ready; no packages were installed by this cell.')

In [ ]:
import shutil, sys, torch
from importlib.metadata import version

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Axolotl:', version('axolotl'))
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is visible. Restart Open OnDemand with a GPU allocation.')
props = torch.cuda.get_device_properties(0)
gpu_memory_gb = props.total_memory / 1024**3
print(f'GPU: {props.name} ({gpu_memory_gb:.1f} GiB)')
print('Free run filesystem:', shutil.disk_usage(RUN_ROOT).free / 1024**3, 'GiB')
if gpu_memory_gb < MIN_GPU_MEMORY_GB:
    raise RuntimeError(f'Need at least {MIN_GPU_MEMORY_GB} GiB for BF16 31B LoRA; requested GPU has {gpu_memory_gb:.1f} GiB.')
print('Preflight passed. Memory pressure can still depend on sequence length and concurrent GPU users.')

In [ ]:
from getpass import getpass
from huggingface_hub import login

# Enter a token with read access only. It is not saved in this notebook.
hf_token = getpass('Hugging Face token (input hidden): ')
login(token=hf_token, add_to_git_credential=False)
del hf_token
print('Authenticated. Confirm that you accepted the Gemma model license on huggingface.co.')

## 3. Prepare the training data

The input file is newline-delimited JSON. A row may contain a `messages` list, or `instruction`, optional `input`, and `output`. The notebook converts every valid row to `system`/`user`/`assistant` messages, removes exact duplicates, and writes reproducible 80/10/10 splits. Do not display or export sensitive text outside approved storage.

**Discuss:** Why must train and test examples be distinct? What kinds of duplicates could make a test result misleading?

In [ ]:
import hashlib, json, random

def normalise_record(row):
    if isinstance(row.get('messages'), list):
        messages = row['messages']
    else:
        instruction, input_text, output = row.get('instruction', ''), row.get('input', ''), row.get('output', '')
        user = '\n\n'.join(part.strip() for part in (instruction, input_text) if isinstance(part, str) and part.strip())
        messages = [{'role': 'user', 'content': user}, {'role': 'assistant', 'content': output}]
    cleaned = []
    for message in messages:
        if not isinstance(message, dict) or message.get('role') not in {'system', 'user', 'assistant'}:
            return None
        content = message.get('content')
        if not isinstance(content, str) or not content.strip():
            return None
        cleaned.append({'role': message['role'], 'content': content.strip()})
    if not any(m['role'] == 'user' for m in cleaned) or cleaned[-1]['role'] != 'assistant':
        return None
    return {'messages': cleaned}

if not SOURCE_DATASET.is_file():
    raise FileNotFoundError(f'Place the approved sanitized JSONL file at: {SOURCE_DATASET}')
records, seen, rejected = [], set(), 0
with SOURCE_DATASET.open(encoding='utf-8') as handle:
    for line_number, line in enumerate(handle, 1):
        try: row = normalise_record(json.loads(line))
        except json.JSONDecodeError: row = None
        if row is None:
            rejected += 1; continue
        key = hashlib.sha256(json.dumps(row, sort_keys=True).encode()).hexdigest()
        if key not in seen:
            seen.add(key); records.append(row)
random.Random(SEED).shuffle(records)
records = records[:MAX_SAMPLES]
if len(records) < 10:
    raise ValueError('Need at least 10 valid, unique records to create meaningful splits.')
n_train, n_val = int(.8 * len(records)), int(.1 * len(records))
splits = {'train': records[:n_train], 'validation': records[n_train:n_train+n_val], 'test': records[n_train+n_val:]}
DATA_DIR = RUN_ROOT / 'data'; DATA_DIR.mkdir(exist_ok=True)
for name, rows in splits.items():
    with (DATA_DIR / f'{name}.jsonl').open('w', encoding='utf-8') as handle:
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False) + '\n')
print({name: len(rows) for name, rows in splits.items()}, '| rejected:', rejected)
print('Schema check only:', [(m['role'], len(m['content'])) for m in splits['train'][0]['messages']])

## Inspect one formatted training example

This is the exact chat-message structure sent to Axolotl. Each record normally has a `system` instruction, a `user` request, and the desired `assistant` answer. Display only approved, de-identified classroom data; do not export examples outside the approved project location.

In [ ]:
# Change this index to inspect a different already-shuffled training example.
SAMPLE_INDEX = 0
example = splits['train'][SAMPLE_INDEX]
print(f'Training example {SAMPLE_INDEX} of {len(splits["train"])}')
print(json.dumps(example, indent=2, ensure_ascii=False))

for message in example['messages']:
    print(f'\n--- {message["role"].upper()} ---')
    print(message['content'])

## 4. Configure Axolotl for BF16 LoRA

The configuration targets language attention and MLP projections only. `load_in_4bit` and `load_in_8bit` are intentionally absent. It uses Transformers **eager attention**, not FlashAttention, so students do not need to compile FlashAttention kernels. Batch size one, gradient accumulation, and gradient checkpointing reduce memory pressure. This notebook uses one fixed classroom configuration.

In [ ]:
import yaml

config = {

    # Hugging Face identifier for the pretrained Gemma checkpoint. If you use your local model point to that path.
    # Hugging Face model ID. Replace MODEL_ID with a local path if the model is pre-staged.
    'base_model': MODEL_ID,

    # Tokenizer and chat template; normally use the same source as the base model.
    'tokenizer_config': MODEL_ID,

    # Gemma 4's end-of-turn token. This tells Axolotl where one chat message ends before the next begins.
    # Keep `<turn|>` for the Gemma 4 chat template; changing it can cause malformed conversations or loss masking errors.
    'eot_tokens': ['<turn|>'],

    # Do not execute arbitrary Python code supplied by the model repository.
    'trust_remote_code': False,

    # Sanitized training JSONL formatted as chat conversations.
    'datasets': [{'path': str(DATA_DIR / 'train.jsonl'), 'type': 'chat_template', 'split': 'train'}],

    # Held-out validation JSONL used while training.
    'test_datasets': [{'path': str(DATA_DIR / 'validation.jsonl'), 'type': 'chat_template', 'split': 'train', 'name': 'validation'}],

    # Folder for adapters and resumable checkpoints.
    'output_dir': str(RUN_ROOT / 'adapter'),

    # Folder for the reusable prepared/tokenized dataset.
    'dataset_prepared_path': str(RUN_ROOT / 'prepared'),

    # Parameter-efficient fine-tuning method; the base model remains frozen.
    'adapter': 'lora',

    # Rank of the LoRA update. Try 8 for a faster/smaller adapter, 16 for this baseline, or 32 for more capacity and VRAM use.
    'lora_r': 16,

    # Scaling factor for the LoRA update. A common pairing is alpha = 2 × rank (for example, 16/32 or 32/64).
    'lora_alpha': 32,

    # Dropout regularization. Try 0.0 for deterministic fitting or 0.05–0.10 when a small dataset appears to overfit.
    'lora_dropout': 0.05,
    # Gemma language attention and MLP projections that receive LoRA adapters.
    'lora_target_modules': '^model\\.language_model\\.layers\\.\\d+\\.(?:self_attn\\.(?:q_proj|k_proj|v_proj|o_proj)|mlp\\.(?:gate_proj|up_proj|down_proj))$',
    
    # Keep adapter weights in BF16 rather than silently upcasting to FP32.
    'peft_autocast_adapter_dtype': False,

    # Use BF16 precision for standard LoRA training.
    'bf16': True,

    # Enable TF32 matrix math on supported NVIDIA hardware.
    'tf32': True,

    # Examples per GPU forward pass. Keep 1 for 31B; try 2 only if GPU memory has substantial headroom.
    'micro_batch_size': 1,

    # Micro-batches accumulated per update. With batch 1, 8 gives an effective batch of 8; try 4 or 16 to change it.
    'gradient_accumulation_steps': 8,

    # Validation batch size.
    'eval_batch_size': 1,

    # Maximum tokens per example. This class uses 1024; 512 is faster, while 2048 captures more context but costs much more VRAM.
    'sequence_len': 1024,

    # Do not concatenate examples into packed sequences.
    'sample_packing': False,

    # Apply the same unpacked behavior during validation.
    'eval_sample_packing': False,

    # Recompute activations in backward pass to reduce VRAM use.
    'gradient_checkpointing': True,
    # Choose the non-reentrant implementation for compatibility.
    'gradient_checkpointing_kwargs': {'use_reentrant': False},

    # Fused AdamW optimizer updates only the LoRA parameters.
    'optimizer': 'adamw_torch_fused',

    # Peak optimizer step size. Try 2e-5 for gentler updates, 5e-5 as the baseline, or 1e-4 cautiously for faster adaptation.
    'learning_rate': 5e-5,

    # Decay the learning rate smoothly over training.
    'lr_scheduler': 'cosine',

    # Fraction of training used to ramp up the learning rate.
    'warmup_ratio': 0.03,

    # AdamW regularization strength.
    'weight_decay': 0.01,

    # Clip gradients to make updates more stable.
    'max_grad_norm': 1.0,

    # Full passes over training data. This class uses 3; try 1 for a short test or 5 only while monitoring validation loss.
    'num_epochs': 3,

    # Stop after 50 optimizer updates for this classroom run. This cap takes precedence over num_epochs.
    # Try 10 for a quick pipeline check or remove this setting to train for all 3 epochs.
    'max_steps': 50,

    # Reproducibility seed.
    'seed': SEED,

    # Calculate loss on assistant responses, not the user prompt.
    'train_on_inputs': False,

    # Evaluate after a fixed number of optimizer steps.
    'eval_strategy': 'steps',

    # Optimizer steps between validation runs.
    'eval_steps': 25,
    # Save checkpoints after a fixed number of optimizer steps.
    'save_strategy': 'steps',

    # Optimizer steps between saved checkpoints.
    'save_steps': 25,

    # Retain only recent checkpoints to conserve project storage.
    'save_total_limit': 2,

    # Frequency of loss/progress messages; 5 provides regular feedback without excessive log output.
    'logging_steps': 5,

    # Save tensors in the safe, non-pickle safetensors format.
    'save_safetensors': True,

    # Retain optimizer state so training can resume after interruption.
    'save_only_model': False,


    # Use eager Transformers attention. It avoids FlashAttention installation; a FlashAttention option would be faster but needs compiled kernels.
    'attn_implementation': 'eager',

    # Keep unused vision/multimodal components frozen in this text-only lesson.
    'freeze_mm_modules': True,

    # Disable Axolotl optimized kernels to avoid extra compilation requirements.
    'use_kernels': False,

    # Disable the optimized QKV LoRA kernel.
    'lora_qkv_kernel': False,

    # Disable the optimized output-projection LoRA kernel.
    'lora_o_kernel': False,

    # Disable the optimized MLP LoRA kernel.
    'lora_mlp_kernel': False,
}

CONFIG_PATH = RUN_ROOT / 'axolotl_lora.yaml'
# The YAML stays clean; the Python comments above are the student-facing explanations.
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

## How to understand the training hyperparameters

The code comments are a quick reference; this section explains what changes when students edit a value. Change **one setting at a time**, then compare runtime, validation loss, and held-out outputs. Paths and `base_model` are infrastructure settings; the controls below change learning.

### LoRA capacity

- `adapter: lora` freezes Gemma's original weights and trains small adapter matrices. `lora_r` is adapter capacity: try `8` for a smaller/faster adapter, `16` for the baseline, or `32` for more capacity and VRAM use. `lora_alpha` scales the update; alpha about twice rank (`16/32`, `32/64`) is a sensible starting point.
- `lora_dropout` is adapter regularization: `0.0` fits data most aggressively, `0.05` is the baseline, and `0.10` is worth trying when validation loss rises while training loss falls. `lora_target_modules` decides which Gemma projections get adapters.

### Memory and effective batch size

- `micro_batch_size` is examples in one GPU forward pass. Keep `1` for BF16 31B; use `2` only with confirmed VRAM headroom. `gradient_accumulation_steps` combines multiple micro-batches: `1 × 8` produces an effective batch size of 8. Try `4` for more frequent/noisier updates or `16` for smoother/slower ones.
- `sequence_len` is maximum tokens per example. This notebook uses `1024`; `512` is faster, while `2048` keeps more context but sharply increases memory and time. `gradient_checkpointing: true` saves memory by recomputing activations, trading VRAM for slower training.
- `bf16: true` reduces memory versus FP32 and requires supported GPUs. `sample_packing: false` favors predictable Gemma 4 behavior over maximum throughput. `attn_implementation: eager` avoids FlashAttention kernel installation, at the cost of speed.

### Learning and regularization

- `learning_rate` controls update size: try `2e-5` for cautious changes, `5e-5` for the baseline, and `1e-4` only while monitoring validation loss. `num_epochs` is passes over the data: this notebook uses `3`; however, `max_steps: 50` stops this classroom run after 50 optimizer updates. Remove that cap to complete all epochs.
- `lr_scheduler: cosine` lowers the learning rate over time; `warmup_ratio: 0.03` avoids large unstable initial updates. `weight_decay`, `lora_dropout`, and `max_grad_norm` limit unstable or overly specific fitting. `train_on_inputs: false` means loss teaches assistant answers rather than user prompts.

### Measurement and recovery

- `eval_steps` controls how often validation is measured: lower values give faster feedback but take more time. `save_steps` controls checkpoint frequency; `save_total_limit: 2` protects storage. Keeping `save_only_model: false` retains optimizer state so interrupted training can resume.

**Suggested exercise:** Run this configuration with `lora_r=8`, then `lora_r=32`, keeping everything else fixed. Compare adapter size, validation loss, and three held-out responses.

## 5. Baseline, training, and evaluation

First generate a few responses from the base model. Then train and compare the adapter on the held-out test split. Loss measures next-token prediction on this split; it cannot prove correctness. Review outputs for unsupported claims, leakage, formatting failures, and unsafe behavior.

**Exercise:** Before training, predict how increasing LoRA rank or epochs might change fit, runtime, and overfitting risk.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def generate_examples(model, tokenizer, examples, label):
    results = []
    for example in examples[:3]:
        prompt_messages = [m for m in example['messages'] if m['role'] != 'assistant']
        prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        results.append({'label': label, 'prompt_messages': prompt_messages, 'generation': text})
    return results

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
baseline = generate_examples(base_model, tokenizer, splits['test'], 'base')
(RUN_ROOT / 'baseline_generations.json').write_text(json.dumps(baseline, indent=2, ensure_ascii=False))
print('Saved baseline generations. Delete the base model before training to free VRAM.')
del base_model
torch.cuda.empty_cache()

In [ ]:
import subprocess

# Re-run this cell after interruption; explicitly resume from the newest checkpoint.
checkpoints = sorted((RUN_ROOT / 'adapter').glob('checkpoint-*'), key=lambda path: int(path.name.rsplit('-', 1)[-1]))
# Jupyter may omit the active Conda environment's bin directory from PATH.
# Resolve Axolotl beside this kernel's Python instead of relying on a global `axolotl` command.
axolotl_executable = Path(sys.executable).with_name('axolotl')
if not axolotl_executable.is_file():
    raise FileNotFoundError(f'Axolotl executable not found beside this kernel: {axolotl_executable}. Select the personal Axolotl kernel and rerun the bootstrap if needed.')
kernel_bin = str(axolotl_executable.parent)
accelerate_executable = Path(kernel_bin) / 'accelerate'
if not accelerate_executable.is_file():
    raise FileNotFoundError(f'Accelerate executable not found beside this kernel: {accelerate_executable}. Rerun the personal-kernel bootstrap if needed.')
# Axolotl launches Accelerate as a child process. Put the personal environment first so it never resolves /apps/jupyter Python 3.10 tools.
training_env = os.environ.copy()
training_env['PATH'] = kernel_bin + os.pathsep + training_env.get('PATH', '')
training_env['PYTHONNOUSERSITE'] = '1'
command = [str(axolotl_executable), 'train', str(CONFIG_PATH)]
if checkpoints:
    command.extend(['--resume_from_checkpoint', str(checkpoints[-1])])
log_path = RUN_ROOT / 'axolotl_train.log'
print('Running:', ' '.join(command))
print('Persistent training log:', log_path)
with log_path.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=training_env)
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
    return_code = process.wait()
if return_code != 0:
    log_tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-60:]
    print('\n--- Final 60 Axolotl log lines ---')
    print('\n'.join(log_tail))
    raise RuntimeError(f'Axolotl exited with code {return_code}. The diagnostic log is saved at {log_path}.')

In [ ]:
from peft import PeftModel

adapter_path = RUN_ROOT / 'adapter'
reloaded_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
tuned_model = PeftModel.from_pretrained(reloaded_base, adapter_path)
tuned = generate_examples(tuned_model, tokenizer, splits['test'], 'lora_adapter')
comparison = [{'base': b, 'lora': t} for b, t in zip(baseline, tuned)]
(RUN_ROOT / 'base_vs_lora.json').write_text(json.dumps(comparison, indent=2, ensure_ascii=False))
summary = {'seed': SEED, 'model_id': MODEL_ID, 'split_sizes': {name: len(rows) for name, rows in splits.items()}, 'baseline_examples': len(baseline), 'adapter_examples': len(tuned)}
(RUN_ROOT / 'evaluation_summary.json').write_text(json.dumps(summary, indent=2))
for item in comparison:
    print('--- BASE ---\n', item['base']['generation'][:500])
    print('--- LORA ---\n', item['lora']['generation'][:500])
print(f'Saved comparison and adapter under {RUN_ROOT}')

## 6. Interpretation, troubleshooting, and extensions

Expected results: this classroom run should log train and validation loss and produce a small LoRA adapter. A lower loss is not automatically better. Inspect held-out generations for factuality, appropriate uncertainty, adherence to response format, and evidence of memorization.

Troubleshooting: model-access errors usually mean the Gemma license/token is missing; CUDA OOM means reduce sequence length or use a larger allocation; package failures often need a kernel restart after installation; malformed JSONL is counted as rejected rows; checkpoints remain in the run directory after an interruption. Never copy sensitive source data or tokens into notebook outputs.

Extensions: compare LoRA ranks 8/16/32, change the number of training examples, add a rubric-based human review, use a separate approved benchmark, or run a longer batch training job. Do not merge the adapter into the base model unless a deployment workflow explicitly needs it; merging requires additional storage and validation.